In [1]:
pip install openrouteservice

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import geopandas as gpd
import os
import openrouteservice
import time
import folium
from itertools import chain
import matplotlib.pyplot as plt

This script serves the purpose of generating a nxn matrix where n = number of planning units. We seek to calculate the travel time driving distance from each planning unit's centroid to the centroid of every other planning unit.

In [9]:
class distanceMatrix:
    def __init__(self, pu_path, key):
        self.pu_path = pu_path
        self.key = key
        self.client = openrouteservice.Client(key = key)

    def load_data(self):
        self.pu = gpd.read_file(f'{os.getcwd()}/data/{self.pu_path}').to_crs('EPSG:4326')
        #self.pu['pu_2324_84'] = self.pu['pu_2324_84'] - 1

    def get_pu_centroids(self):
        self.pu['centroid'] = self.pu.geometry.centroid
        self.centroids = self.pu['centroid']

    def isochrone_branching(self, pu = 1, time = 600):      #isochrone branches for pu argument
        centroid = self.centroids.iloc[pu]
        coords = (centroid.x,centroid.y)
        self.iso = self.client.isochrones(
            locations = [coords],
            profile = 'driving-car',
            range = [time]
        )      

        self.iso_geo = gpd.GeoDataFrame.from_features(self.iso['features'], crs = 'EPSG:4326')['geometry']

    def distance_array(self, pu = 1, max_time = 1800, step = 60):
        N = len(self.centroids)
        self.times = [None] * N
        range1 = range(step, int(2*max_time/3) + step, step)               #loop every minute between 1 and 10 min.
        range2 = range(int(2*max_time/3) + step, max_time + step, step*2)  #loop every other minute from 11 to 30 min.
        range_all = chain(range1, range2)
        for t in range_all:                                   #branch isochrones by step
            self.isochrone_branching(pu = pu, time = t)
            isochrone = self.iso_geo.union_all()             #unary_union -> union_all?

            #get what centroids lay within isochrone
            overlap = self.pu[self.pu.intersects(isochrone)]
            indices = overlap.index.to_list()

            #enter new times into list
            for idx in indices:
                if self.times[idx] is None:
                    self.times[idx] = int(t/60)
            
            #break if all centroids have been reached
            if all(time is not None for time in self.times):
                break

            #sleep to avoid surpassing 40/minute quota
            time.sleep(1.5)

    def build(self, pu_lower = 0, pu_upper = 0):
        self.matrix_times = pd.DataFrame()
        for unit in range(pu_lower, pu_upper + 1):
            self.distance_array(pu = unit)                    #get distance array for individual pu
            col = self.times
            self.matrix_times[unit] = col

        self.matrix_times.to_csv('dist_matrix_0_80.csv')
    

In [10]:
matrix = distanceMatrix('pu_split_start_0.geojson', 'eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6ImRmZGMwZDA3NDVhYzRkNzY5Y2UzN2Q1YTk3MmNlNWQzIiwiaCI6Im11cm11cjY0In0=')
matrix.load_data()
matrix.get_pu_centroids()
#matrix.isochrone_branching()
#matrix.distance_array()
matrix.build(pu_lower = 0, pu_upper = 80)

C:\Users\olubl\AppData\Local\Temp\ipykernel_13732\3454620025.py:12: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  self.pu['centroid'] = self.pu.geometry.centroid
C:\Users\olubl\anaconda3\envs\spatialdata\Lib\site-packages\openrouteservice\client.py:211: UserWarning: Rate limit exceeded. Retrying for the 1st time.
  warnings.warn('Rate limit exceeded. Retrying for the {0}{1} time.'.format(retry_counter + 1,


In [15]:
# m = folium.Map((35.900,-78.882), tiles="OpenStreetMap", zoom_start = 10)
# folium.GeoJson(matrix.iso_geo).add_to(m)
# folium.GeoJson(matrix.pu.loc[1].geometry).add_to(m)
# m

In [72]:
#take average over all diagonals?

In [73]:
# time_map = gpd.GeoDataFrame(
#     {
#     'times' : matrix.times,
#     'geometry' : matrix.pu.geometry
#     }
# ).fillna(60).set_crs('EPSG:4326')

In [74]:
# time_map.plot(column = 'times')